# 🌍 District Accord — GRPO Training Notebook

Train `Qwen2.5-1.5B-Instruct` to play as a district agent in the District Accord
multi-agent environment using **GRPO** (Group Relative Policy Optimization).

**Setup:**
- Runtime: GPU (T4 or better) — Runtime → Change runtime type → T4 GPU
- Environment: 12 districts, 100 turns per episode
- Framework: TRL + Unsloth

**Flow:**
1. Install dependencies
2. Load model (Qwen2.5-1.5B-Instruct, 4-bit via Unsloth)
3. Launch environment (12 agents, 100 turns)
4. Define observation → prompt formatter
5. Collect training prompts from episodes
6. Define multi-component reward function
7. Run GRPO training
8. Generate reward/loss plots
9. Test trained model live
10. Save and download results

In [ ]:
# ============================================================
# CELL 1: Install Dependencies
# ============================================================
# unsloth   — 4-bit model loading + LoRA training efficiency
# trl       — GRPOTrainer for RL training
# datasets  — HuggingFace Dataset used as training data format
# matplotlib — plot generation for reward curves
# The project itself is installed with -e (editable) so
# district_accord can be imported directly.

!pip install unsloth trl datasets matplotlib --quiet
!git clone https://github.com/tezivindh/meta-env-hackathon-final.git
%cd meta-env-hackathon-final
!pip install -e . --quiet
print('✅ Dependencies installed')

In [ ]:
# ============================================================
# CELL 2: Load Model with Unsloth
# ============================================================
# We use Qwen2.5-1.5B-Instruct — already instruction-tuned,
# so we skip SFT and go straight to RL (GRPO).
#
# load_in_4bit=True  → quantize weights to 4-bit for memory efficiency
# LoRA r=16          → low-rank adaptation, trains ~1% of parameters
# target_modules     → which attention/MLP layers to adapt
# use_gradient_checkpointing='unsloth' → Unsloth's memory-efficient variant

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

# Ensure pad token is set (required by TRL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('✅ Model loaded: Qwen2.5-1.5B-Instruct (4-bit + LoRA r=16)')

In [ ]:
# ============================================================
# CELL 3: Setup Environment
# ============================================================
# DistrictAccordEnv follows the Gym-style API:
#   obs = env.reset(seed=42)
#   obs, rewards, done, trunc, info = env.step(actions)
#
# EnvConfig holds all hyperparameters:
#   num_districts=12  → 12 competing/cooperating agents
#   max_turns=100     → long-horizon (100 decision steps per episode)
#
# ActionParser converts raw text strings (e.g. 'invest', 'defend')
# into structured ParsedAction dictionaries the env understands.
#
# SelfPlayPolicy provides: random | mask_aware_random | rule_based
# agents that we use as opponents for the LLM to train against.

from district_accord.env import DistrictAccordEnv
from district_accord.utils.config import EnvConfig
from district_accord.spaces.action_parser import ActionParser
from district_accord.policy.self_play import SelfPlayPolicy
import numpy as np

# Full 12-district, 100-turn configuration
train_cfg = EnvConfig(num_districts=12, max_turns=100)

# Instantiate environment and parser
env = DistrictAccordEnv(train_cfg)
parser = ActionParser(train_cfg)

# Quick sanity check
obs = env.reset(seed=42)
print('✅ Environment loaded')
print(f'   Agents: {list(obs.keys())}')
print(f'   Observation keys per agent: {list(obs[0].keys())}')
print(f'   Action mask shape: {obs[0]["action_mask"].shape}  (9 possible actions)')

# Verify step works end-to-end
opponent = SelfPlayPolicy(mode='rule_based', seed=42)
actions = opponent.act(obs, env)
obs2, rewards, done, trunc, info = env.step(actions)
print(f'   Step OK — sample reward for agent 0: {rewards[0]:.4f}')

In [ ]:
# ============================================================
# CELL 4: Define Observation → Prompt Formatter
# ============================================================
# The LLM doesn't receive raw numpy arrays — it receives a
# natural language prompt describing its current state.
#
# SYSTEM_PROMPT: defines the task, action space, and output format
# obs_to_prompt(): converts one agent's observation dict into
#   a turn-specific state summary the LLM can reason about.
#
# Key design choices:
#   - Include valid actions dynamically from the action mask
#     (avoids the model learning to output masked/invalid actions)
#   - Show peer averages (not full 11-agent breakdown) to keep
#     prompt length manageable
#   - Include crisis level prominently — it's the main signal
#     for whether to defend/invest/cooperate

VALID_ACTIONS = [
    "invest", "defend", "ignore", "recover",
    "request_aid", "share", "propose", "accept", "reject",
]

SYSTEM_PROMPT = """You are a district in a multi-agent crisis management environment with 12 districts.
Each turn, choose exactly ONE action to maximize survival and cooperation over 100 turns.

Actions:
- invest: Grow resources, mild stability boost
- defend: Boost stability, reduce crisis exposure (costs resources)
- recover: Emergency stability recovery (costs more resources)
- ignore: Do nothing (passive drains still apply)
- propose: Propose coalition with another district
- accept: Accept a pending coalition proposal
- reject: Reject a pending coalition proposal
- share: Share resources with another district
- request_aid: Request aid from another district

Reply with ONLY the action name. Nothing else."""

# The LLM-controlled agent is always agent 0
AGENT_ID = 0


def obs_to_prompt(obs_dict: dict, agent_id: int, env: DistrictAccordEnv) -> str:
    """
    Convert an agent's observation dictionary into a natural language prompt.

    Args:
        obs_dict: observation for this agent from env.reset() or env.step()
        agent_id: integer agent ID (used for display only)
        env: live environment reference (read-only, used for config)

    Returns:
        Formatted prompt string to append after SYSTEM_PROMPT.
    """
    s = obs_dict["self"]        # [resources, stability, crisis_exposure, stability_delta]
    c = obs_dict["crisis"]      # [crisis_level]
    t = obs_dict["turn"]        # [turn_progress 0..1]
    mask = obs_dict["action_mask"]  # [9] binary — 1 = action is valid this turn
    others = obs_dict["others"] # list of [resources, stability, trust, in_coalition] per peer

    # Aggregate peer state into summary statistics
    avg_peer_res  = float(np.mean([o[0] for o in others])) if len(others) > 0 else 0
    avg_peer_stab = float(np.mean([o[1] for o in others])) if len(others) > 0 else 0

    # Only show actions that are valid this turn
    valid = [VALID_ACTIONS[i] for i, m in enumerate(mask)
             if m == 1 and i < len(VALID_ACTIONS)]

    # Current turn number (denormalized from [0,1])
    turn_num = int(t[0] * env.config.max_turns)

    return (
        f"Turn {turn_num}/{env.config.max_turns} | Crisis Level: {c[0]:.2f}\n"
        f"My Resources: {s[0]:.3f} | My Stability: {s[1]:.3f} "
        f"| Exposure: {s[2]:.3f} | Stability Delta: {s[3]:+.3f}\n"
        f"Avg Peer Resources: {avg_peer_res:.3f} | Avg Peer Stability: {avg_peer_stab:.3f}\n"
        f"Valid actions: {', '.join(valid)}\n\n"
        f"Your action:"
    )


# Show a sample prompt so you can see what the model receives
env_test = DistrictAccordEnv(train_cfg)
obs_test = env_test.reset(seed=42)
sample_prompt = SYSTEM_PROMPT + "\n\n" + obs_to_prompt(obs_test[0], 0, env_test)
print('Sample prompt that the LLM receives each turn:')
print('=' * 60)
print(sample_prompt)
print('=' * 60)

In [ ]:
# ============================================================
# CELL 5: Collect Training Prompts
# ============================================================
# We run episodes using the rule_based opponent for ALL agents
# (including agent 0) just to collect prompts — we don't care
# about actions here, only the observations at each turn.
#
# Each prompt represents one decision point for agent 0.
# GRPO will later sample multiple completions per prompt,
# score them, and update the model.
#
# With 20 episodes × up to 100 turns = up to 2000 prompts.

from datasets import Dataset


def collect_prompts(cfg: EnvConfig, num_episodes: int = 20, seed: int = 42) -> list:
    """
    Run episodes and collect observation prompts for agent 0.

    Args:
        cfg: environment configuration
        num_episodes: number of episodes to collect from
        seed: base random seed (each episode uses seed + episode_index)

    Returns:
        List of dicts with key 'prompt' — ready for Dataset.from_list()
    """
    all_prompts = []
    opponent = SelfPlayPolicy(mode="rule_based", seed=seed)

    for ep in range(num_episodes):
        ep_seed = seed + ep
        env = DistrictAccordEnv(cfg)
        obs = env.reset(seed=ep_seed)
        opponent._rng = np.random.default_rng(ep_seed)  # deterministic per episode

        for turn in range(cfg.max_turns):
            if env._done:
                break

            # Record the prompt for agent 0 at this turn
            full_prompt = SYSTEM_PROMPT + "\n\n" + obs_to_prompt(
                obs[AGENT_ID], AGENT_ID, env
            )
            all_prompts.append({"prompt": full_prompt})

            # Advance the episode using rule_based for all agents
            actions = opponent.act(obs, env)
            obs, _, done, trunc, _ = env.step(actions)
            if done or trunc:
                break

        if (ep + 1) % 5 == 0:
            print(f'  Episode {ep + 1}/{num_episodes} — total prompts so far: {len(all_prompts)}')

    return all_prompts


print('Collecting training prompts...')
prompts = collect_prompts(train_cfg, num_episodes=20, seed=42)
dataset = Dataset.from_list(prompts)
print(f'\n✅ Training dataset ready: {len(dataset)} prompts')
print(f'   Each prompt = one decision turn for agent 0 in a 12-district episode')

In [ ]:
# ============================================================
# CELL 6: Define Reward Function
# ============================================================
# This is what GRPO uses to score each completion.
# Multiple independent checks reduce reward hacking risk
# (an agent can't just game one signal).
#
# Reward components (all additive):
#   +0.5  valid action name
#   +0.2  clean format (single token, no extra text)
#   +0.4  context-appropriate: defend/recover in high crisis
#   +0.3  low stability: recover is critical
#   +0.2  good resources + stable: invest bonus
#   +0.3  cooperative actions: propose/accept/share
#   -0.2  ignore when action is available (passive hurts)
#   -0.3  multi-word response (bad format)
#   -1.0  invalid action (not in VALID_ACTIONS)


def district_reward_fn(prompts: list, completions: list, **kwargs) -> list:
    """
    Score each (prompt, completion) pair for GRPO training.

    Called by GRPOTrainer after each generation batch.
    Returns a list of scalar rewards, one per completion.
    """
    rewards = []

    for prompt, completion in zip(prompts, completions):
        # Extract the action text from the completion
        if hasattr(completion, 'text'):
            action = completion.text.strip().split("\n")[0].strip().lower()
        elif isinstance(completion, list):  # token IDs
            action = tokenizer.decode(completion, skip_special_tokens=True
                                      ).strip().split("\n")[0].strip().lower()
        else:
            action = str(completion).strip().split("\n")[0].strip().lower()

        score = 0.0
        base_action = action.split(":")[0]  # handle 'share:target=2' etc.

        # Hard gate: invalid action gets maximum penalty and stops
        if base_action not in VALID_ACTIONS:
            rewards.append(-1.0)
            continue

        # ── Component 1: Valid action ──────────────────────────────
        score += 0.5

        # ── Component 2: Format compliance ────────────────────────
        # Good: 'defend' or 'share:target=2'   Bad: 'I will defend now'
        if len(action.split()) <= 2:
            score += 0.2
        else:
            score -= 0.3

        # ── Components 3-6: Context-aware scoring ──────────────────
        try:
            # Parse key state values from the prompt text
            crisis_val, resources, stability = 0.0, 0.5, 0.5
            for line in prompt.split("\n"):
                if "Crisis Level:" in line:
                    crisis_val = float(line.split("Crisis Level:")[-1].strip())
                if "My Resources:" in line:
                    for part in line.split("|"):
                        if "My Resources:" in part:
                            resources = float(part.split(":")[-1].strip())
                        elif "My Stability:" in part:
                            stability = float(part.split(":")[-1].strip())

            # Component 3: High crisis → defend/recover is correct
            if crisis_val > 0.4 and base_action in ("defend", "recover"):
                score += 0.4

            # Component 4: Low stability → recovery is critical
            if stability < 0.3 and base_action == "recover":
                score += 0.3

            # Component 5: Stable + resourced → invest is optimal
            if resources > 0.4 and stability > 0.5 and base_action == "invest":
                score += 0.2

            # Component 6: Cooperative actions are always encouraged
            if base_action in ("propose", "accept", "share"):
                score += 0.3

            # Penalty: ignore when action is not forced
            if base_action == "ignore":
                score -= 0.2

        except (ValueError, IndexError):
            pass  # If parsing fails, skip context scoring

        rewards.append(score)

    return rewards


# Sanity check
test_p = ["Turn 10/100 | Crisis Level: 0.6\nMy Resources: 0.4 | My Stability: 0.2",
          "Turn 10/100 | Crisis Level: 0.1\nMy Resources: 0.7 | My Stability: 0.8"]
test_c = ["recover", "invest"]
test_r = district_reward_fn(test_p, test_c)
print('✅ Reward function test')
print(f'   recover (high crisis, low stability) → {test_r[0]:.2f}  (expected: high)')
print(f'   invest  (low crisis, high stability) → {test_r[1]:.2f}  (expected: high)')

In [ ]:
# ============================================================
# CELL 7: GRPO Training
# ============================================================
# GRPOConfig key parameters:
#   num_generations=4     → generate 4 completions per prompt,
#                           rank them by reward, update model
#   max_completion_length → action strings are short (8-16 tokens max)
#   max_prompt_length     → cap prompt length (avoids OOM on long prompts)
#   per_device_train_batch_size=2 + gradient_accumulation_steps=2
#                         → effective batch size of 4 (T4 memory safe)
#   num_train_epochs=2    → 2 passes over the dataset
#   learning_rate=5e-6    → conservative lr (RL is sensitive to large steps)
#   fp16=True             → T4 supports fp16 but not bf16
# ============================================================

from trl import GRPOConfig, GRPOTrainer
import os

os.makedirs("outputs", exist_ok=True)
os.makedirs("outputs/grpo_checkpoints", exist_ok=True)

training_args = GRPOConfig(
    output_dir="outputs/grpo_checkpoints",
    num_generations=4,             # completions to sample per prompt
    max_completion_length=16,      # action strings are very short
    max_prompt_length=1800,        # truncate long prompts
    per_device_train_batch_size=2, # prompts per GPU step
    gradient_accumulation_steps=2, # effective batch = 2×2 = 4
    num_train_epochs=2,            # passes over full dataset
    learning_rate=5e-6,            # conservative — RL is sensitive
    logging_steps=5,               # log every 5 steps
    save_steps=200,                # checkpoint every 200 steps
    report_to="none",              # disable wandb/tensorboard
    fp16=True,                     # T4 supports fp16 (not bf16)
    seed=42,
)

trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    reward_funcs=[district_reward_fn],  # our multi-component scorer
    args=training_args,
    train_dataset=dataset,
)

print('🚀 Starting GRPO training...')
print(f'   Dataset size: {len(dataset)} prompts')
print(f'   Epochs: {training_args.num_train_epochs}')
print(f'   Generations per prompt: {training_args.num_generations}')
print(f'   Total completions: ~{len(dataset) * training_args.num_generations * training_args.num_train_epochs:,}')
print()

trainer.train()
print('\n✅ Training complete!')

In [ ]:
# ============================================================
# CELL 8: Generate Training Plots
# ============================================================
# Produces two plots saved to outputs/:
#
# training_curves.png
#   Left:  GRPO training loss over steps
#   Right: Mean reward per logging step, with baseline references
#
# baseline_comparison.png
#   Bar chart: Random vs Mask-Aware vs Rule-Based vs Trained LLM
#
# These are embedded directly in the README and HF Space.

import matplotlib.pyplot as plt

# Extract log history from trainer state
log_history = trainer.state.log_history
steps, losses, rews = [], [], []
for entry in log_history:
    if "loss" in entry:
        steps.append(entry.get("step", len(steps)))
        losses.append(entry["loss"])
    if "reward" in entry:
        rews.append(entry["reward"])

# ── Plot 1: Training loss + reward curve ──────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("District Accord — GRPO Training (12 Districts, 100 Turns)", fontsize=13)

if losses:
    ax1.plot(steps[:len(losses)], losses, color="#60a5fa", lw=2)
    ax1.set(xlabel="Training Step", ylabel="Loss", title="GRPO Training Loss")
    ax1.grid(alpha=0.3)
else:
    ax1.text(0.5, 0.5, 'No loss data', transform=ax1.transAxes, ha='center')

if rews:
    ax2.plot(range(len(rews)), rews, color="#6ee7b7", lw=2, label="GRPO (trained)")
    # Reference lines from our measured baselines
    ax2.axhline(0.397, color="#f87171", ls="--", lw=1.5, label="Random baseline (0.397)")
    ax2.axhline(1.002, color="#fbbf24", ls="--", lw=1.5, label="Rule-based ceiling (1.002)")
    ax2.set(xlabel="Training Step", ylabel="Reward",
            title="Reward During GRPO Training")
    ax2.legend(fontsize=9)
    ax2.grid(alpha=0.3)
else:
    ax2.text(0.5, 0.5, 'No reward data', transform=ax2.transAxes, ha='center')

plt.tight_layout()
plt.savefig("outputs/training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print('✅ Saved outputs/training_curves.png')

# ── Plot 2: Policy comparison bar chart ───────────────────────────────────
# Use actual measured baselines + final trained reward
trained_reward = float(np.mean(rews[-10:])) if len(rews) >= 10 else 0.9

fig, ax = plt.subplots(figsize=(9, 5))
policies = ["Random", "Mask-Aware\nRandom", "Rule-Based", "Trained LLM\n(GRPO)"]
avg_rewards = [0.397, 0.939, 1.002, trained_reward]
colors = ["#f87171", "#fbbf24", "#6ee7b7", "#60a5fa"]

bars = ax.bar(policies, avg_rewards, color=colors, width=0.55,
              edgecolor="white", linewidth=1.5)
ax.set_ylabel("Avg Reward / Turn / Agent", fontsize=11)
ax.set_title("District Accord — Policy Comparison (12 Districts)", fontsize=12)
ax.set_ylim(0, 1.4)
ax.grid(axis="y", alpha=0.3)

for bar_obj, val in zip(bars, avg_rewards):
    ax.text(bar_obj.get_x() + bar_obj.get_width() / 2,
            bar_obj.get_height() + 0.03,
            f"{val:.3f}", ha="center", fontweight="bold", fontsize=11)

plt.tight_layout()
plt.savefig("outputs/baseline_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print('✅ Saved outputs/baseline_comparison.png')

In [ ]:
# ============================================================
# CELL 9: Test Trained Model — Live Episode
# ============================================================
# This runs a full episode where:
#   - Agent 0 = trained LLM (generates text actions)
#   - Agents 1-11 = rule_based opponents
#
# We print each turn's action and reward so you can see
# qualitatively whether the model learned sensible behavior.
#
# FastLanguageModel.for_inference() switches the model from
# training mode to fast generation mode (Unsloth optimization).

from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)  # enable fast inference mode

# Fresh episode with new seed
test_env = DistrictAccordEnv(train_cfg)
obs = test_env.reset(seed=999)
opponent = SelfPlayPolicy(mode="rule_based", seed=999)
total_reward = 0.0
action_counts = {}

print('🎮 Trained LLM playing as District 0 (12 agents, 100 turns)\n')
print(f'{"Turn":>5} | {"Action":15} | {"Reward":>8} | Cumulative')
print('-' * 50)

for turn in range(train_cfg.max_turns):
    if test_env._done:
        break

    # Format prompt for the LLM
    prompt = SYSTEM_PROMPT + "\n\n" + obs_to_prompt(obs[AGENT_ID], AGENT_ID, test_env)

    # Generate action (greedy-ish with low temperature)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=8,     # action strings are short
        temperature=0.3,      # low temp = more deterministic
        do_sample=True,
    )
    # Decode only the generated tokens (exclude the prompt)
    action = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip().split("\n")[0].lower()

    # Count actions for final summary
    action_counts[action] = action_counts.get(action, 0) + 1

    # Parse LLM action and get opponent actions
    llm_parsed = parser.parse_structured_safe({AGENT_ID: action})
    opp_actions = opponent.act(obs, env)
    all_actions = {**opp_actions, **llm_parsed}

    # Step environment
    obs, rewards, done, trunc, info = test_env.step(all_actions)
    r = rewards.get(AGENT_ID, 0)
    total_reward += r

    # Print every 10 turns to avoid flooding output
    if turn % 10 == 0 or turn < 5:
        print(f'{turn:>5} | {action:15} | {r:>+8.3f} | {total_reward:.2f}')

    if done or trunc:
        break

print('-' * 50)
print(f'\n📊 Results (agent 0 = trained LLM):')
print(f'   Turns survived: {turn + 1}/{train_cfg.max_turns}')
print(f'   Total reward:   {total_reward:.3f}')
print(f'   Avg reward/turn: {total_reward / (turn + 1):.4f}')
print(f'   Action distribution: {dict(sorted(action_counts.items(), key=lambda x: -x[1]))}')

In [ ]:
# ============================================================
# CELL 10: Save Model and Download Results
# ============================================================
# save_method='lora' saves only the LoRA adapter weights (~50MB)
# This is correct for 4-bit + LoRA models (do NOT use
# merged_16bit on a 4-bit base — that damages quality).
#
# outputs/ directory will contain:
#   training_curves.png      — loss + reward curve
#   baseline_comparison.png  — policy comparison bar chart
#   district_accord_lora/    — trained LoRA adapter
#   grpo_checkpoints/        — intermediate checkpoints

import os

# Save LoRA adapter (small, reproducible)
save_path = "outputs/district_accord_lora"
model.save_pretrained_merged(save_path, tokenizer, save_method="lora")
print(f'✅ LoRA adapter saved to {save_path}/')

# List everything in outputs/
print('\nOutputs directory:')
for f in sorted(os.listdir('outputs')):
    path = os.path.join('outputs', f)
    size = os.path.getsize(path) if os.path.isfile(path) else '-'
    print(f'  {f:40} {size}')

# Zip everything and download
!zip -r outputs.zip outputs/ --quiet
print('\n✅ Zipped to outputs.zip')

from google.colab import files
files.download('outputs.zip')
print('✅ Download started!')